<a href="https://colab.research.google.com/github/MBR4V0/Python/blob/main/Treli%C3%A7a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import math

class TrussMember:

    def __init__(self, force_kN, length_m, area_cm2, E_GPa=200, fy_MPa=250, k_factor=1.0):

        self.F = force_kN * 1000  # N
        self.L = length_m
        self.A = area_cm2 / 10000  # m²
        self.E = E_GPa * 1e9       # Pa
        self.fy = fy_MPa * 1e6     # Pa
        self.K = k_factor

    # -----------------------------
    # Tensão atuante
    # -----------------------------
    def stress(self):
        return self.F / self.A

    # -----------------------------
    # Resistência à tração (NBR 8800)
    # -----------------------------
    def tensile_capacity(self):
        return 0.9 * self.fy * self.A

    # -----------------------------
    # Inércia aproximada (barra maciça simplificada)
    # I ≈ A² / 12 (aproximação grosseira)
    # -----------------------------
    def inertia(self):
        return (self.A ** 2) / 12

    # -----------------------------
    # Carga crítica de flambagem (Euler)
    # -----------------------------
    def euler_buckling(self):
        return (math.pi ** 2 * self.E * self.inertia()) / ((self.K * self.L) ** 2)

    # -----------------------------
    # Verificação de compressão
    # -----------------------------
    def compression_check(self):

        Ncr = self.euler_buckling()

        if self.F > 0:  # compressão
            if self.F >= Ncr:
                return "❌ COLAPSO por flambagem (Euler atingido)", Ncr
            else:
                if self.F >= 0.5 * Ncr:
                    return "⚠ Risco de flambagem alto", Ncr
                else:
                    return "✔ Seguro contra flambagem", Ncr
        else:
            return "✔ Tração (sem flambagem)", Ncr

    # -----------------------------
    # Verificação de tração
    # -----------------------------
    def tension_check(self):

        Ft = abs(self.F)
        capacity = self.tensile_capacity()

        if Ft > capacity:
            return "❌ COLAPSO por ruptura à tração", capacity
        else:
            return "✔ Seguro à tração", capacity

    # -----------------------------
    # Diagnóstico final
    # -----------------------------
    def analyze(self):

        print("\n📐 VERIFICAÇÃO DE TRELIÇA - ABNT NBR 8800 (SIMPLIFICADO)")
        print("=" * 70)

        stress = self.stress() / 1e6  # MPa

        print(f"Força axial: {self.F/1000:.2f} kN")
        print(f"Tensão: {stress:.2f} MPa")

        # tração
        t_status, t_cap = self.tension_check()

        # flambagem
        c_status, Ncr = self.compression_check()

        print("\n🔹 TRAÇÃO:")
        print(t_status)
        print(f"Capacidade ≈ {t_cap/1000:.2f} kN")

        print("\n🔹 COMPRESSÃO / FLAMBAGEM:")
        print(c_status)
        print(f"Ncr (Euler) ≈ {Ncr/1000:.2f} kN")

        print("\n📌 CLASSIFICAÇÃO FINAL:")

        if "COLAPSO" in t_status or "COLAPSO" in c_status:
            print("❌ ELEMENTO INVIÁVEL")
        elif "Risco" in c_status:
            print("⚠ ELEMENTO CRÍTICO (reforçar seção)")
        else:
            print("✔ ELEMENTO SEGURADO SEGUNDO CRITÉRIOS SIMPLIFICADOS DA NBR 8800")


# -----------------------------
# EXEMPLO
# -----------------------------
if __name__ == "__main__":

    # força (kN), comprimento (m), área (cm²)
    bar = TrussMember(
        force_kN=-120,   # compressão
        length_m=3.0,
        area_cm2=10.0
    )

    bar.analyze()


📊 RELATÓRIO NBR 6122 (SIMPLIFICADO)
Média SPT: 11.67
Classificação: Consistência média
Risco de recalque: Recalque moderado

Tensão aplicada: 120.00 kPa
Tensão admissível (σadm): 87.50 kPa

🔎 VERIFICAÇÃO FUNDAÇÃO RASA:
❌ NÃO viável para fundação rasa
👉 Sugestão: fundações profundas (estacas ou tubulões)

📌 OBS:
- Método simplificado baseado em SPT
- Deve ser validado com sondagem completa e projeto geotécnico
